# Experiment: K-Means Clustering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## Task 1: K-Means (K=2) from scratch on the given points

In [ ]:
data = np.array([(185,72),(170,56),(168,60),(179,68),(182,72),(188,77)], dtype=float)
K = 2
centroids = data[:2].copy()   # first two points as initial centroids

for it in range(1, 11):
    # Assignment: Euclidean distance to each centroid
    dist = np.linalg.norm(data[:, None] - centroids[None], axis=2)
    labels = dist.argmin(axis=1)
    print(f"--- Iteration {it} ---")
    print("Distances:\n", np.round(dist, 2))
    for k in range(K):
        print(f"Cluster {k+1}:", data[labels == k].tolist())
    # Update: new centroids = mean of each cluster
    new_centroids = np.array([data[labels == k].mean(axis=0) for k in range(K)])
    print("New centroids:", np.round(new_centroids, 2).tolist(), "\n")
    if np.allclose(new_centroids, centroids):
        print("Converged.")
        break
    centroids = new_centroids

plt.scatter(data[:,0], data[:,1], c=labels, cmap='viridis', s=80)
plt.scatter(centroids[:,0], centroids[:,1], c='red', marker='X', s=200, label='Centroids')
plt.xlabel('Height'); plt.ylabel('Weight'); plt.title('K-Means (K=2) from scratch')
plt.legend(); plt.show()

Verification with scikit-learn:

In [ ]:
km = KMeans(n_clusters=2, init=data[:2], n_init=1).fit(data)
print("Labels:", km.labels_)
print("Centroids:\n", km.cluster_centers_)

## Task 2: K-Means on the Iris dataset

In [ ]:
# Option A: built-in Iris (no upload needed)
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
df = iris.frame
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))

# Option B: if you have IRIS.csv, upload it to Colab and use instead:
# df = pd.read_csv('IRIS.csv')   # then adjust column names below

features = df.iloc[:, :4]   # sepal/petal length & width
df.head()

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
df['cluster'] = kmeans.fit_predict(features)

print("Centroids:\n", pd.DataFrame(kmeans.cluster_centers_, columns=features.columns).round(2))
print("\nCluster vs actual species:")
print(pd.crosstab(df['cluster'], df['species']))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
x, y = features.columns[2], features.columns[3]   # petal length vs width
ax[0].scatter(df[x], df[y], c=df['cluster'], cmap='viridis')
ax[0].scatter(kmeans.cluster_centers_[:,2], kmeans.cluster_centers_[:,3], c='red', marker='X', s=200)
ax[0].set_title('K-Means clusters'); ax[0].set_xlabel(x); ax[0].set_ylabel(y)
ax[1].scatter(df[x], df[y], c=df['species'].astype('category').cat.codes, cmap='viridis')
ax[1].set_title('Actual species'); ax[1].set_xlabel(x); ax[1].set_ylabel(y)
plt.show()

## Curiosity: choosing the optimal K (Elbow + Silhouette)

In [ ]:
ks = range(2, 11)
inertia, sil = [], []
for k in ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(features)
    inertia.append(m.inertia_)
    sil.append(silhouette_score(features, m.labels_))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ks, inertia, 'o-'); ax[0].set_title('Elbow method'); ax[0].set_xlabel('K'); ax[0].set_ylabel('Inertia (WCSS)')
ax[1].plot(ks, sil, 'o-'); ax[1].set_title('Silhouette score'); ax[1].set_xlabel('K'); ax[1].set_ylabel('Score')
plt.show()
print("Best K by silhouette:", list(ks)[int(np.argmax(sil))])